<a href="https://colab.research.google.com/github/ionikim/epilepsy_pediatrics_EEG/blob/main/src/01_loading/graph_to_gephi_type.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [7]:
import pandas as pd
from pathlib import Path

In [8]:
edge_list_ictal = pd.read_parquet(
    "data/graphs/edge_lists/chb01_03_window_ictal_0_edge_list.parquet"
)

node_table_ictal = pd.read_parquet(
    "data/graphs/node_tables/chb01_03_window_ictal_0_nodes.parquet"
)

print(edge_list_ictal.shape)
print(node_table_ictal.shape)

display(edge_list_ictal.head())
display(node_table_ictal.head())

(381641, 6)
(29440, 5)


,source,target,source_label,target_label,edge_type,layer
0,819,820,FP1-F7_t819,FP1-F7_t820,intra,FP1-F7
1,820,821,FP1-F7_t820,FP1-F7_t821,intra,FP1-F7
2,818,819,FP1-F7_t818,FP1-F7_t819,intra,FP1-F7
3,821,822,FP1-F7_t821,FP1-F7_t822,intra,FP1-F7
4,817,818,FP1-F7_t817,FP1-F7_t818,intra,FP1-F7


,node_id,electrode,time_idx,time_s,node_label
0,0,FP1-F7,0,2996.000000,FP1-F7_t0
1,1,FP1-F7,1,2996.003906,FP1-F7_t1
2,2,FP1-F7,2,2996.007812,FP1-F7_t2
3,3,FP1-F7,3,2996.011719,FP1-F7_t3
4,4,FP1-F7,4,2996.015625,FP1-F7_t4


In [9]:
edge_list_interictal = pd.read_parquet(
    "data/graphs/edge_lists/chb01_03_window_interictal_0_edge_list.parquet"
)

node_table_interictal = pd.read_parquet(
    "data/graphs/node_tables/chb01_03_window_interictal_0_nodes.parquet"
)

print(edge_list_interictal.shape)
print(node_table_interictal.shape)

(381667, 6)
(29440, 5)


In [10]:
def export_to_gephi(edge_list, node_table, graph_id, out_dir="data/gephi", max_edges=None):
    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)

    # ----- Edges -----
    gephi_edges = edge_list.copy()

    gephi_edges = gephi_edges.rename(columns={
        "source": "Source",
        "target": "Target"
    })

    # Gephi edge type : undirected
    gephi_edges["Type"] = "Undirected"

    # when we don't have weight, it will assign 1.0
    if "weight" in gephi_edges.columns:
        gephi_edges = gephi_edges.rename(columns={"weight": "Weight"})
    else:
        gephi_edges["Weight"] = 1.0

    # edge type store
    if "edge_type" in gephi_edges.columns:
        gephi_edges["edge_type"] = gephi_edges["edge_type"]

    # when it's too big, keep the part of it
    if max_edges is not None and len(gephi_edges) > max_edges:
        gephi_edges = gephi_edges.sample(n=max_edges, random_state=42)

    edge_cols = ["Source", "Target", "Type", "Weight"]
    extra_edge_cols = [c for c in ["edge_type", "layer", "source_label", "target_label"] if c in gephi_edges.columns]
    edge_cols += extra_edge_cols

    edge_path = out_dir / f"{graph_id}_edges.csv"
    gephi_edges[edge_cols].to_csv(edge_path, index=False)

    # ----- Nodes -----
    gephi_nodes = node_table.copy()

    gephi_nodes = gephi_nodes.rename(columns={
        "node_id": "Id",
        "node_label": "Label"
    })

    node_cols = ["Id", "Label"]
    extra_node_cols = [c for c in ["electrode", "time_idx", "time_s"] if c in gephi_nodes.columns]
    node_cols += extra_node_cols

    node_path = out_dir / f"{graph_id}_nodes.csv"
    gephi_nodes[node_cols].to_csv(node_path, index=False)

    print(f"Saved Gephi edges: {edge_path}")
    print(f"Saved Gephi nodes: {node_path}")

    return edge_path, node_path

In [11]:
export_to_gephi(
    edge_list=edge_list_ictal,
    node_table=node_table_ictal,
    graph_id="chb01_03_ictal",
    out_dir="data/gephi"
)

Saved Gephi edges: data/gephi/chb01_03_ictal_edges.csv
Saved Gephi nodes: data/gephi/chb01_03_ictal_nodes.csv


(PosixPath('data/gephi/chb01_03_ictal_edges.csv'),
 PosixPath('data/gephi/chb01_03_ictal_nodes.csv'))

In [12]:
export_to_gephi(
    edge_list=edge_list_ictal,
    node_table=node_table_ictal,
    graph_id="chb01_03_ictal_small",
    out_dir="data/gephi",
    max_edges=5000
)

Saved Gephi edges: data/gephi/chb01_03_ictal_small_edges.csv
Saved Gephi nodes: data/gephi/chb01_03_ictal_small_nodes.csv


(PosixPath('data/gephi/chb01_03_ictal_small_edges.csv'),
 PosixPath('data/gephi/chb01_03_ictal_small_nodes.csv'))

In [13]:
export_to_gephi(
    edge_list=edge_list_interictal,
    node_table=node_table_interictal,
    graph_id="chb01_03_interictal",
    out_dir="data/gephi"
)

export_to_gephi(
    edge_list=edge_list_interictal,
    node_table=node_table_interictal,
    graph_id="chb01_03_interictal_small",
    out_dir="data/gephi",
    max_edges=5000
)

Saved Gephi edges: data/gephi/chb01_03_interictal_edges.csv
Saved Gephi nodes: data/gephi/chb01_03_interictal_nodes.csv
Saved Gephi edges: data/gephi/chb01_03_interictal_small_edges.csv
Saved Gephi nodes: data/gephi/chb01_03_interictal_small_nodes.csv


(PosixPath('data/gephi/chb01_03_interictal_small_edges.csv'),
 PosixPath('data/gephi/chb01_03_interictal_small_nodes.csv'))